# Credora Finance — Model Building & Evaluation

## Objective

Build and evaluate machine learning models to predict loan default using the feature-engineered Credora Finance dataset.

Models:
- Logistic Regression
- Decision Tree
- Random Forest
- XGBoost

Evaluation will focus on:
- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC
- Confusion Matrix

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv('/content/credora_modeling_dataset.csv')

print("Dataset loaded successfully.")
print("=" * 50)

print("Shape:", df.shape)

print("\nTarget distribution:")
print(df['loan_default'].value_counts())

print("\nTarget percentage:")
print(df['loan_default'].value_counts(normalize=True).mul(100).round(2))

Dataset loaded successfully.
Shape: (3403, 42)

Target distribution:
loan_default
0    2606
1     797
Name: count, dtype: int64

Target percentage:
loan_default
0    76.58
1    23.42
Name: proportion, dtype: float64


## 1. Feature-Target Separation

In [4]:
# Separate input features and target variable

X = df.drop(columns=['loan_default'])
y = df['loan_default']

print("Feature matrix (X):", X.shape)
print("Target vector (y):", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentage:")
print(y.value_counts(normalize=True).mul(100).round(2))

Feature matrix (X): (3403, 41)
Target vector (y): (3403,)

Target distribution:
loan_default
0    2606
1     797
Name: count, dtype: int64

Target percentage:
loan_default
0    76.58
1    23.42
Name: proportion, dtype: float64


In [5]:
# Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("TRAIN-TEST SPLIT")
print("=" * 50)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nTraining target percentage:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTesting target percentage:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

TRAIN-TEST SPLIT
X_train: (2722, 41)
X_test : (681, 41)
y_train: (2722,)
y_test : (681,)

Training target percentage:
loan_default
0    76.56
1    23.44
Name: proportion, dtype: float64

Testing target percentage:
loan_default
0    76.65
1    23.35
Name: proportion, dtype: float64


## 2. Preprocessing

In [6]:
# Identify numerical and categorical features

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include='object').columns.tolist()

print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

print("\nNUMERICAL FEATURES")
print("-" * 50)
print(numeric_features)

print("\nCATEGORICAL FEATURES")
print("-" * 50)
print(categorical_features)

Number of numerical features: 30
Number of categorical features: 11

NUMERICAL FEATURES
--------------------------------------------------
['requested_amount', 'loan_term_months', 'interest_rate', 'applicant_monthly_income', 'existing_emi', 'debt_to_income_ratio', 'collateral_flag', 'annual_income', 'loan_to_income', 'age', 'dependents', 'years_at_residence', 'credit_score', 'num_open_accounts', 'num_credit_inquiries_6m', 'num_late_payments_30d', 'num_late_payments_90d', 'num_defaults_prior', 'bankruptcies', 'total_credit_limit', 'total_outstanding_debt', 'oldest_account_age_months', 'debt_to_limit', 'total_inflow_6m', 'total_outflow_6m', 'emi_outflow_6m', 'avg_balance', 'transaction_count', 'net_cash_flow_6m', 'inflow_outflow_ratio']

CATEGORICAL FEATURES
--------------------------------------------------
['loan_type', 'loan_purpose', 'gender', 'marital_status', 'education', 'employment_type', 'occupation', 'city', 'state', 'residence_type', 'kyc_status']


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [8]:
# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine numerical and categorical preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


## 3. Logistic Regression

In [9]:
from sklearn.linear_model import LogisticRegression

# Create pipeline: preprocessing + Logistic Regression
logistic_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train the model
logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [10]:
# Predict classes and default probabilities on the test set

y_pred_logistic = logistic_model.predict(X_test)

y_prob_logistic = logistic_model.predict_proba(X_test)[:, 1]

print("Predictions created successfully.")
print("Number of predictions:", len(y_pred_logistic))

print("\nPredicted class counts:")
print(pd.Series(y_pred_logistic).value_counts().sort_index())

Predictions created successfully.
Number of predictions: 681

Predicted class counts:
0    623
1     58
Name: count, dtype: int64


In [11]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Calculate evaluation metrics
logistic_accuracy = accuracy_score(y_test, y_pred_logistic)
logistic_precision = precision_score(y_test, y_pred_logistic)
logistic_recall = recall_score(y_test, y_pred_logistic)
logistic_f1 = f1_score(y_test, y_pred_logistic)
logistic_roc_auc = roc_auc_score(y_test, y_prob_logistic)

print("LOGISTIC REGRESSION PERFORMANCE")
print("=" * 50)

print(f"Accuracy : {logistic_accuracy:.4f}")
print(f"Precision: {logistic_precision:.4f}")
print(f"Recall   : {logistic_recall:.4f}")
print(f"F1-Score : {logistic_f1:.4f}")
print(f"ROC-AUC  : {logistic_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_logistic))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_logistic))

LOGISTIC REGRESSION PERFORMANCE
Accuracy : 0.7695
Precision: 0.5172
Recall   : 0.1887
F1-Score : 0.2765
ROC-AUC  : 0.7059

Confusion Matrix:
[[494  28]
 [129  30]]

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.95      0.86       522
           1       0.52      0.19      0.28       159

    accuracy                           0.77       681
   macro avg       0.66      0.57      0.57       681
weighted avg       0.73      0.77      0.73       681



### Logistic Regression — Key Findings

- Logistic Regression achieved **76.95% accuracy** and a **ROC-AUC of 0.7059**.
- The model achieved **51.72% precision** for the default class but only **18.87% recall**.
- Out of **159 actual default cases**, the model correctly identified only **30** and missed **129**.
- The high number of false negatives indicates that the baseline Logistic Regression model is weak at identifying borrowers who actually default.
- Although overall accuracy is relatively high, it is influenced by the larger number of non-default cases in the dataset.
- The model does not currently meet the project targets of **ROC-AUC ≥ 0.80** and **Recall ≥ 0.70**.
- This model will be retained as a baseline and compared with Decision Tree, Random Forest, and XGBoost before model tuning and final selection.

## 4. Decision Tree

In [12]:
from sklearn.tree import DecisionTreeClassifier

# Create pipeline: preprocessing + Decision Tree
decision_tree_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(
        random_state=42
    ))
])

# Train the model
decision_tree_model.fit(X_train, y_train)

print("Decision Tree trained successfully.")

Decision Tree trained successfully.


In [13]:
# Make predictions
y_pred_dt = decision_tree_model.predict(X_test)
y_prob_dt = decision_tree_model.predict_proba(X_test)[:, 1]

# Calculate metrics
dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_precision = precision_score(y_test, y_pred_dt)
dt_recall = recall_score(y_test, y_pred_dt)
dt_f1 = f1_score(y_test, y_pred_dt)
dt_roc_auc = roc_auc_score(y_test, y_prob_dt)

print("DECISION TREE PERFORMANCE")
print("=" * 50)

print(f"Accuracy : {dt_accuracy:.4f}")
print(f"Precision: {dt_precision:.4f}")
print(f"Recall   : {dt_recall:.4f}")
print(f"F1-Score : {dt_f1:.4f}")
print(f"ROC-AUC  : {dt_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

DECISION TREE PERFORMANCE
Accuracy : 0.6769
Precision: 0.3129
Recall   : 0.3208
F1-Score : 0.3168
ROC-AUC  : 0.5531

Confusion Matrix:
[[410 112]
 [108  51]]

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.79      0.79       522
           1       0.31      0.32      0.32       159

    accuracy                           0.68       681
   macro avg       0.55      0.55      0.55       681
weighted avg       0.68      0.68      0.68       681



### Decision Tree — Key Findings

- Decision Tree achieved **67.69% accuracy** and a **ROC-AUC of 0.5531**.
- Default-class precision was **31.29%**, while recall was **32.08%**.
- Out of **159 actual default cases**, the model correctly identified **51** and missed **108**.
- Recall improved compared with the baseline Logistic Regression model, but this improvement came with substantially more false-positive predictions.
- The ROC-AUC of **0.5531** indicates weak overall discrimination between default and non-default cases.
- The baseline Decision Tree does not meet the project targets of **ROC-AUC ≥ 0.80** and **Recall ≥ 0.70**.
- The model will be retained for comparison with the remaining baseline models before tuning and final model selection.

## 5. Random Forest

In [14]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

random_forest_model.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [15]:
# Make predictions
y_pred_rf = random_forest_model.predict(X_test)
y_prob_rf = random_forest_model.predict_proba(X_test)[:, 1]

# Calculate metrics
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_roc_auc = roc_auc_score(y_test, y_prob_rf)

print("RANDOM FOREST PERFORMANCE")
print("=" * 50)

print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1-Score : {rf_f1:.4f}")
print(f"ROC-AUC  : {rf_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

RANDOM FOREST PERFORMANCE
Accuracy : 0.7636
Precision: 0.4667
Recall   : 0.0881
F1-Score : 0.1481
ROC-AUC  : 0.6798

Confusion Matrix:
[[506  16]
 [145  14]]

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.97      0.86       522
           1       0.47      0.09      0.15       159

    accuracy                           0.76       681
   macro avg       0.62      0.53      0.51       681
weighted avg       0.70      0.76      0.70       681



### Random Forest — Key Findings

- Random Forest achieved **76.36% accuracy** and a **ROC-AUC of 0.6798**.
- Default-class precision was **46.67%**, while recall was only **8.81%**.
- Out of **159 actual default cases**, the model correctly identified only **14** and missed **145**.
- The model performed very well on the non-default class but struggled substantially to identify default cases.
- The relatively high overall accuracy is influenced by the class imbalance, since non-default cases represent the majority of the dataset.
- The baseline Random Forest does not meet the project targets of **ROC-AUC ≥ 0.80** and **Recall ≥ 0.70**.
- Class imbalance handling and model tuning may be required after all baseline models are compared.

## 6. XGBoost

In [16]:
from xgboost import XGBClassifier

xgboost_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=200,
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1
    ))
])

xgboost_model.fit(X_train, y_train)

print("XGBoost trained successfully.")

XGBoost trained successfully.


In [17]:
# Make predictions
y_pred_xgb = xgboost_model.predict(X_test)
y_prob_xgb = xgboost_model.predict_proba(X_test)[:, 1]

# Calculate metrics
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
xgb_precision = precision_score(y_test, y_pred_xgb)
xgb_recall = recall_score(y_test, y_pred_xgb)
xgb_f1 = f1_score(y_test, y_pred_xgb)
xgb_roc_auc = roc_auc_score(y_test, y_prob_xgb)

print("XGBOOST PERFORMANCE")
print("=" * 50)

print(f"Accuracy : {xgb_accuracy:.4f}")
print(f"Precision: {xgb_precision:.4f}")
print(f"Recall   : {xgb_recall:.4f}")
print(f"F1-Score : {xgb_f1:.4f}")
print(f"ROC-AUC  : {xgb_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

XGBOOST PERFORMANCE
Accuracy : 0.7327
Precision: 0.3814
Recall   : 0.2327
F1-Score : 0.2891
ROC-AUC  : 0.6361

Confusion Matrix:
[[462  60]
 [122  37]]

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.89      0.84       522
           1       0.38      0.23      0.29       159

    accuracy                           0.73       681
   macro avg       0.59      0.56      0.56       681
weighted avg       0.70      0.73      0.71       681



### XGBoost — Key Findings

- XGBoost achieved **73.27% accuracy** and a **ROC-AUC of 0.6361**.
- Default-class precision was **38.14%**, while recall was **23.27%**.
- Out of **159 actual default cases**, the model correctly identified **37** and missed **122**.
- XGBoost identified more default cases than Logistic Regression and Random Forest, but fewer than the Decision Tree.
- Its ROC-AUC was lower than Logistic Regression and Random Forest, indicating weaker overall discrimination in the baseline configuration.
- The baseline XGBoost model does not meet the project targets of **ROC-AUC ≥ 0.80** and **Recall ≥ 0.70**.
- Further evaluation using stratified cross-validation and class-imbalance handling is required before tuning and final model selection.

## 7. Stratified 5-Fold Cross-Validation

To obtain a more reliable estimate of model performance, 5-fold stratified
cross-validation is performed on the training dataset. Stratification preserves
the default/non-default class distribution within each fold.

In [18]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

print("5-fold Stratified Cross-Validation configured successfully.")

5-fold Stratified Cross-Validation configured successfully.


In [19]:
models = {
    'Logistic Regression': logistic_model,
    'Decision Tree': decision_tree_model,
    'Random Forest': random_forest_model,
    'XGBoost': xgboost_model
}

cv_results = []

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results.append({
        'Model': name,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1-Score': scores['test_f1'].mean(),
        'ROC-AUC': scores['test_roc_auc'].mean()
    })

cv_results_df = pd.DataFrame(cv_results)

print("5-FOLD CROSS-VALIDATION RESULTS")
print("=" * 75)
print(cv_results_df.round(4).to_string(index=False))

5-FOLD CROSS-VALIDATION RESULTS
              Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC
Logistic Regression    0.7649     0.4967  0.2352    0.3182   0.7156
      Decision Tree    0.6738     0.3198  0.3463    0.3325   0.5601
      Random Forest    0.7579     0.4366  0.1035    0.1663   0.7163
            XGBoost    0.7381     0.4040  0.2398    0.3008   0.6879


### Cross-Validation — Key Findings

- Logistic Regression achieved a mean **ROC-AUC of 0.7156** with **23.52% recall**.
- Random Forest achieved the highest mean **ROC-AUC of 0.7163**, but its recall was very low at **10.35%**.
- Decision Tree achieved the highest recall at **34.63%**, but its ROC-AUC was only **0.5601**, indicating weak discrimination.
- XGBoost achieved a **ROC-AUC of 0.6879** and recall of **23.98%**.
- None of the baseline models currently meets the project targets of **ROC-AUC ≥ 0.80** and **Recall ≥ 0.70**.
- The results indicate that class imbalance is affecting the models' ability to identify default cases.
- Therefore, the next stage will focus on class-imbalance handling and model tuning using only the training data.

## 8. Class Imbalance Handling

The baseline models showed low recall for the default class. Since identifying
potential defaulters is important in credit-risk assessment, class weighting is
applied to give greater importance to the minority default class.

Performance is evaluated using 5-fold stratified cross-validation on the
training data.

In [20]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Non-default training cases:", negative_count)
print("Default training cases:", positive_count)
print("Class imbalance ratio:", round(scale_pos_weight, 4))

Non-default training cases: 2084
Default training cases: 638
Class imbalance ratio: 3.2665


In [21]:
balanced_models = {
    'Logistic Regression Balanced': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=42
        ))
    ]),

    'Decision Tree Balanced': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier(
            class_weight='balanced',
            random_state=42
        ))
    ]),

    'Random Forest Balanced': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=200,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        ))
    ]),

    'XGBoost Balanced': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(
            n_estimators=200,
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            eval_metric='logloss',
            n_jobs=-1
        ))
    ])
}

print("Balanced models created successfully.")

Balanced models created successfully.


In [22]:
balanced_cv_results = []

for name, model in balanced_models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    balanced_cv_results.append({
        'Model': name,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1-Score': scores['test_f1'].mean(),
        'ROC-AUC': scores['test_roc_auc'].mean()
    })

balanced_cv_results_df = pd.DataFrame(balanced_cv_results)

print("BALANCED MODEL — 5-FOLD CROSS-VALIDATION RESULTS")
print("=" * 85)
print(balanced_cv_results_df.round(4).to_string(index=False))

BALANCED MODEL — 5-FOLD CROSS-VALIDATION RESULTS
                       Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC
Logistic Regression Balanced    0.6752     0.3807  0.6162    0.4698   0.7164
      Decision Tree Balanced    0.6837     0.3427  0.3809    0.3606   0.5787
      Random Forest Balanced    0.7630     0.4814  0.0784    0.1336   0.7131
            XGBoost Balanced    0.7267     0.3950  0.3072    0.3453   0.6966


### Class Imbalance Handling — Key Findings

- Applying class weighting substantially improved the ability of Logistic Regression to identify default cases.
- Balanced Logistic Regression achieved **61.62% recall**, compared with **23.52%** for the baseline model.
- Its F1-score improved from **0.3182 to 0.4698**, while ROC-AUC remained stable at approximately **0.7164**.
- This improvement came with a reduction in overall accuracy and precision, which reflects the trade-off involved in detecting more high-risk borrowers.
- Balanced Decision Tree achieved **38.09% recall** and **0.5787 ROC-AUC**, providing only a modest improvement over its baseline version.
- Balanced Random Forest continued to perform poorly on the default class, with only **7.84% recall**.
- Balanced XGBoost improved recall from **23.98% to 30.72%**, but remained substantially below Balanced Logistic Regression.
- Among the balanced models, Logistic Regression currently provides the strongest recall and F1-score while maintaining competitive ROC-AUC.
- However, the project targets of **Recall ≥ 0.70** and **ROC-AUC ≥ 0.80** have not yet been achieved, so further model tuning and threshold optimization will be explored.

## 9. Hyperparameter Tuning

Based on cross-validation results, Balanced Logistic Regression showed the
strongest improvement in default detection. Hyperparameter tuning is therefore
performed using the training data to improve generalization while avoiding
test-set leakage.

In [23]:
from sklearn.model_selection import GridSearchCV

logistic_tuning_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        random_state=42
    ))
])

logistic_param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100]
}

logistic_grid = GridSearchCV(
    estimator=logistic_tuning_pipeline,
    param_grid=logistic_param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

logistic_grid.fit(X_train, y_train)

print("LOGISTIC REGRESSION TUNING RESULTS")
print("=" * 50)
print("Best parameters:", logistic_grid.best_params_)
print("Best CV ROC-AUC:", round(logistic_grid.best_score_, 4))

LOGISTIC REGRESSION TUNING RESULTS
Best parameters: {'classifier__C': 0.01}
Best CV ROC-AUC: 0.7402


### Logistic Regression Tuning Result

- Hyperparameter tuning selected **C = 0.01** as the best regularization parameter.
- The tuned Balanced Logistic Regression achieved a mean cross-validation **ROC-AUC of 0.7402**, improving over the previous balanced baseline ROC-AUC of **0.7164**.
- The tuned model will be further evaluated using cross-validation metrics before final test-set evaluation.

In [24]:
best_logistic_model = logistic_grid.best_estimator_

print("Best tuned Logistic Regression model saved successfully.")

Best tuned Logistic Regression model saved successfully.


In [25]:
tuned_logistic_scores = cross_validate(
    best_logistic_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("TUNED BALANCED LOGISTIC REGRESSION — 5-FOLD CV")
print("=" * 60)

print(f"Accuracy : {tuned_logistic_scores['test_accuracy'].mean():.4f}")
print(f"Precision: {tuned_logistic_scores['test_precision'].mean():.4f}")
print(f"Recall   : {tuned_logistic_scores['test_recall'].mean():.4f}")
print(f"F1-Score : {tuned_logistic_scores['test_f1'].mean():.4f}")
print(f"ROC-AUC  : {tuned_logistic_scores['test_roc_auc'].mean():.4f}")

TUNED BALANCED LOGISTIC REGRESSION — 5-FOLD CV
Accuracy : 0.6914
Precision: 0.4028
Recall   : 0.6552
F1-Score : 0.4986
ROC-AUC  : 0.7402


### Tuned Logistic Regression — Key Findings

- The tuned Balanced Logistic Regression achieved a mean cross-validation **ROC-AUC of 0.7402**.
- Recall improved to **65.52%**, compared with **61.62%** for the balanced baseline and **23.52%** for the original Logistic Regression model.
- The F1-score improved to **0.4986**, showing a better balance between precision and recall.
- Precision was **40.28%**, while accuracy was **69.14%**.
- Hyperparameter tuning therefore improved both default detection and overall discrimination.
- Recall is approaching the project target of **70%**, although ROC-AUC remains below the target of **0.80**.
- Threshold optimization will next be evaluated using training-set cross-validated probabilities to improve recall without using the held-out test set.

## 10. Classification Threshold Optimization

The default classification threshold of 0.50 may not provide the best trade-off
for credit-risk detection. Cross-validated probabilities from the training data
are used to evaluate alternative thresholds without using the held-out test set.

In [26]:
from sklearn.model_selection import cross_val_predict

oof_probabilities = cross_val_predict(
    best_logistic_model,
    X_train,
    y_train,
    cv=cv,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

print("Out-of-fold probabilities generated successfully.")
print("Number of probabilities:", len(oof_probabilities))

Out-of-fold probabilities generated successfully.
Number of probabilities: 2722


In [27]:
threshold_results = []

for threshold in np.arange(0.20, 0.61, 0.05):

    predictions = (oof_probabilities >= threshold).astype(int)

    threshold_results.append({
        'Threshold': round(threshold, 2),
        'Precision': precision_score(y_train, predictions),
        'Recall': recall_score(y_train, predictions),
        'F1-Score': f1_score(y_train, predictions)
    })

threshold_results_df = pd.DataFrame(threshold_results)

print("THRESHOLD COMPARISON")
print("=" * 55)
print(threshold_results_df.round(4).to_string(index=False))

THRESHOLD COMPARISON
 Threshold  Precision  Recall  F1-Score
      0.20     0.2540  0.9843    0.4039
      0.25     0.2672  0.9561    0.4177
      0.30     0.2881  0.9373    0.4407
      0.35     0.3093  0.8793    0.4576
      0.40     0.3431  0.8292    0.4853
      0.45     0.3796  0.7508    0.5042
      0.50     0.4027  0.6552    0.4988
      0.55     0.4287  0.5517    0.4825
      0.60     0.4792  0.4702    0.4747


### Threshold Optimization — Key Findings

- The default threshold of **0.50** produced **65.52% recall** and an F1-score of **0.4988**.
- Lowering the classification threshold increased the model's ability to identify potential defaulters.
- A threshold of **0.45** achieved **75.08% recall**, exceeding the project's target of **Recall ≥ 0.70**.
- At this threshold, precision was **37.96%** and the F1-score was **0.5042**, which was the highest F1-score among the evaluated thresholds.
- Thresholds below 0.45 produced higher recall but substantially lower precision, increasing the number of false-positive default predictions.
- Therefore, **0.45 was selected as the operating threshold** for the tuned Logistic Regression model.
- The threshold was selected entirely from cross-validated training predictions, keeping the held-out test dataset separate from threshold selection.

## 11. Final Model Evaluation

The tuned Balanced Logistic Regression model with a classification threshold
of 0.45 is evaluated on the held-out test dataset.

The test dataset was not used for hyperparameter or threshold selection.

In [28]:
final_model = best_logistic_model
final_threshold = 0.45

# Probability of default on held-out test data
final_probabilities = final_model.predict_proba(X_test)[:, 1]

# Apply selected threshold
final_predictions = (final_probabilities >= final_threshold).astype(int)

print("Final predictions generated successfully.")
print("Selected threshold:", final_threshold)

print("\nPredicted class counts:")
print(pd.Series(final_predictions).value_counts().sort_index())

Final predictions generated successfully.
Selected threshold: 0.45

Predicted class counts:
0    367
1    314
Name: count, dtype: int64


In [29]:
final_accuracy = accuracy_score(y_test, final_predictions)
final_precision = precision_score(y_test, final_predictions)
final_recall = recall_score(y_test, final_predictions)
final_f1 = f1_score(y_test, final_predictions)
final_roc_auc = roc_auc_score(y_test, final_probabilities)

final_cm = confusion_matrix(y_test, final_predictions)

print("FINAL MODEL — HELD-OUT TEST PERFORMANCE")
print("=" * 55)

print(f"Accuracy : {final_accuracy:.4f}")
print(f"Precision: {final_precision:.4f}")
print(f"Recall   : {final_recall:.4f}")
print(f"F1-Score : {final_f1:.4f}")
print(f"ROC-AUC  : {final_roc_auc:.4f}")

print("\nConfusion Matrix:")
print(final_cm)

print("\nClassification Report:")
print(classification_report(y_test, final_predictions))

FINAL MODEL — HELD-OUT TEST PERFORMANCE
Accuracy : 0.6314
Precision: 0.3535
Recall   : 0.6981
F1-Score : 0.4693
ROC-AUC  : 0.7119

Confusion Matrix:
[[319 203]
 [ 48 111]]

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.61      0.72       522
           1       0.35      0.70      0.47       159

    accuracy                           0.63       681
   macro avg       0.61      0.65      0.59       681
weighted avg       0.75      0.63      0.66       681



### Final Model — Key Findings

- The final tuned Balanced Logistic Regression model was evaluated on the held-out test set using the selected **0.45 classification threshold**.
- The model achieved **63.14% accuracy**, **35.35% precision**, **69.81% recall**, an **F1-score of 0.4693**, and a **ROC-AUC of 0.7119**.
- Out of **159 actual default cases**, the model correctly identified **111** and missed **48**.
- The model identified substantially more defaulters than the original Logistic Regression baseline, which detected only 30 of 159 default cases at the default 0.50 threshold.
- The improved recall comes with a higher number of false positives: **203 non-default cases were classified as potential defaults**.
- The final recall of **69.81%** is very close to, but does not technically meet, the project target of **Recall ≥ 70%**.
- The final ROC-AUC of **0.7119** does not meet the project target of **ROC-AUC ≥ 0.80**.
- These results demonstrate a trade-off between identifying more potential defaulters and maintaining precision.
- The final model and threshold were selected using training-set cross-validation, while the held-out test set was reserved for final evaluation.

## 12. Probability of Default and Risk Score

The final model's predicted default probability is converted into a 0–100
risk score.

Higher scores indicate a higher estimated probability of loan default.

Initial risk categories:
- **Low Risk:** 0–33
- **Medium Risk:** 34–66
- **High Risk:** 67–100

In [30]:
# Create final test result dataset
risk_results = X_test.copy()

risk_results['actual_default'] = y_test.values
risk_results['default_probability'] = final_probabilities

# Convert probability to 0–100 risk score
risk_results['risk_score'] = (
    risk_results['default_probability'] * 100
).round().astype(int)

# Assign risk categories
risk_results['risk_category'] = pd.cut(
    risk_results['risk_score'],
    bins=[-1, 33, 66, 100],
    labels=['Low', 'Medium', 'High']
)

print("Risk scores generated successfully.")
print("\nRisk Score Summary:")
print(risk_results['risk_score'].describe())

print("\nRisk Category Distribution:")
print(risk_results['risk_category'].value_counts())

print("\nSample Results:")
print(
    risk_results[
        ['actual_default',
         'default_probability',
         'risk_score',
         'risk_category']
    ].head(10)
)

Risk scores generated successfully.

Risk Score Summary:
count    681.000000
mean      44.110132
std       18.773602
min        2.000000
25%       29.000000
50%       43.000000
75%       57.000000
max       94.000000
Name: risk_score, dtype: float64

Risk Category Distribution:
risk_category
Medium    370
Low       218
High       93
Name: count, dtype: int64

Sample Results:
      actual_default  default_probability  risk_score risk_category
2002               0             0.303488          30           Low
2632               1             0.727953          73          High
194                1             0.557499          56        Medium
1831               1             0.327463          33           Low
191                0             0.370404          37        Medium
2719               0             0.599301          60        Medium
371                0             0.133037          13           Low
75                 1             0.746598          75          High
2198      

### Risk Scoring — Key Findings

- Probability of default was successfully converted into a **0–100 risk score** for all **681 held-out test applications**.
- Risk scores ranged from **2 to 94**, with a median score of **43** and a mean score of approximately **44.11**.
- Based on the defined risk bands, **218 applications** were classified as Low Risk, **370 as Medium Risk, and 93 as High Risk**.
- Medium Risk was the largest category, representing the majority of evaluated applications.
- Higher risk scores represent higher model-estimated probabilities of default.
- These risk scores provide a business-friendly representation of the model output that can be used for portfolio monitoring and decision-support dashboards.

In [31]:
risk_validation = (
    risk_results
    .groupby('risk_category', observed=False)
    .agg(
        applications=('actual_default', 'count'),
        actual_defaults=('actual_default', 'sum'),
        average_risk_score=('risk_score', 'mean'),
        actual_default_rate=('actual_default', 'mean')
    )
)

risk_validation['actual_default_rate'] = (
    risk_validation['actual_default_rate'] * 100
).round(2)

risk_validation['average_risk_score'] = (
    risk_validation['average_risk_score'].round(2)
)

print("RISK CATEGORY VALIDATION")
print("=" * 65)
print(risk_validation)

RISK CATEGORY VALIDATION
               applications  actual_defaults  average_risk_score  \
risk_category                                                      
Low                     218               23               23.04   
Medium                  370               91               48.66   
High                     93               45               75.41   

               actual_default_rate  
risk_category                       
Low                          10.55  
Medium                       24.59  
High                         48.39  


### Risk Category Validation — Key Findings

- The observed default rate increased consistently across the model-defined risk categories.
- **Low Risk** applications had an actual default rate of **10.55%**.
- **Medium Risk** applications had an actual default rate of **24.59%**.
- **High Risk** applications had an actual default rate of **48.39%**.
- The High Risk group therefore showed substantially greater observed default incidence than the Low and Medium Risk groups.
- Average risk scores also increased consistently from **23.04 (Low)** to **48.66 (Medium)** and **75.41 (High)**.
- This monotonic increase indicates that the generated risk score provides useful separation between lower-risk and higher-risk applications.

## 13. Model Explainability with SHAP

SHAP is used to explain the final model's predictions and identify which
features contribute most strongly to estimated default risk.

Both global feature importance and individual prediction explanations are
examined.

In [32]:
# Extract fitted preprocessing and classifier components
fitted_preprocessor = final_model.named_steps['preprocessor']
fitted_classifier = final_model.named_steps['classifier']

# Transform training and test data
X_train_transformed = fitted_preprocessor.transform(X_train)
X_test_transformed = fitted_preprocessor.transform(X_test)

# Get transformed feature names
feature_names = fitted_preprocessor.get_feature_names_out()

print("Training transformed shape:", X_train_transformed.shape)
print("Test transformed shape:", X_test_transformed.shape)
print("Number of transformed features:", len(feature_names))

Training transformed shape: (2722, 135)
Test transformed shape: (681, 135)
Number of transformed features: 135


In [33]:
print("First 30 transformed features:")
for feature in feature_names[:30]:
    print(feature)

First 30 transformed features:
num__requested_amount
num__loan_term_months
num__interest_rate
num__applicant_monthly_income
num__existing_emi
num__debt_to_income_ratio
num__collateral_flag
num__annual_income
num__loan_to_income
num__age
num__dependents
num__years_at_residence
num__credit_score
num__num_open_accounts
num__num_credit_inquiries_6m
num__num_late_payments_30d
num__num_late_payments_90d
num__num_defaults_prior
num__bankruptcies
num__total_credit_limit
num__total_outstanding_debt
num__oldest_account_age_months
num__debt_to_limit
num__total_inflow_6m
num__total_outflow_6m
num__emi_outflow_6m
num__avg_balance
num__transaction_count
num__net_cash_flow_6m
num__inflow_outflow_ratio


In [34]:
# Install/import SHAP if necessary
try:
    import shap
except ImportError:
    !pip install -q shap
    import shap

# Convert sparse matrices to dense arrays if required
if hasattr(X_train_transformed, "toarray"):
    X_train_shap = X_train_transformed.toarray()
    X_test_shap = X_test_transformed.toarray()
else:
    X_train_shap = X_train_transformed
    X_test_shap = X_test_transformed

# Create SHAP explainer for Logistic Regression
explainer = shap.LinearExplainer(
    fitted_classifier,
    X_train_shap
)

# Calculate SHAP values for test data
shap_values = explainer(X_test_shap)

print("SHAP values generated successfully.")
print("SHAP values shape:", shap_values.values.shape)

SHAP values generated successfully.
SHAP values shape: (681, 135)


In [35]:
shap_importance = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': np.abs(shap_values.values).mean(axis=0)
})

shap_importance = (
    shap_importance
    .sort_values('mean_abs_shap', ascending=False)
    .reset_index(drop=True)
)

print("TOP 20 FEATURES BY MEAN ABSOLUTE SHAP VALUE")
print("=" * 75)
print(shap_importance.head(20).to_string(index=False))

TOP 20 FEATURES BY MEAN ABSOLUTE SHAP VALUE
                           feature  mean_abs_shap
         num__debt_to_income_ratio       0.333173
                 num__credit_score       0.225718
        num__num_late_payments_30d       0.121993
      num__num_credit_inquiries_6m       0.097096
             num__requested_amount       0.095584
     num__applicant_monthly_income       0.090205
                num__annual_income       0.090081
                num__interest_rate       0.082034
             num__loan_term_months       0.075769
     cat__employment_type_Salaried       0.071860
       num__total_outstanding_debt       0.070981
cat__employment_type_Self-Employed       0.057661
      cat__education_Post Graduate       0.054092
             num__total_outflow_6m       0.049576
              num__collateral_flag       0.047749
           cat__education_Graduate       0.046707
            num__num_open_accounts       0.045567
               num__loan_to_income       0.041615
      

### Global SHAP — Key Findings

- SHAP analysis was successfully performed on the final Logistic Regression model across **135 transformed features**.
- **Debt-to-income ratio** was the most influential feature globally, with the highest mean absolute SHAP value (**0.3332**).
- **Credit score** was the second most influential feature (**0.2257**).
- Credit-behavior variables such as **30-day late payments** and **credit inquiries in the previous 6 months** were also among the strongest contributors.
- Loan characteristics including **requested amount, interest rate, and loan term** contributed meaningfully to model predictions.
- Income-related variables, including **applicant monthly income** and **annual income**, were also important.
- Employment type, education, outstanding debt, transaction outflow, collateral status, loan-to-income ratio, and debt-to-limit ratio also appeared among the top global predictors.
- Mean absolute SHAP values measure the magnitude of a feature's influence and do not by themselves indicate whether the feature increases or decreases predicted default risk.

In [36]:
# Find the test-set position with the highest predicted default probability
high_risk_position = np.argmax(final_probabilities)

print("Selected test-set position:", high_risk_position)
print("Default probability:", round(final_probabilities[high_risk_position], 4))
print("Risk score:", risk_results.iloc[high_risk_position]['risk_score'])
print("Risk category:", risk_results.iloc[high_risk_position]['risk_category'])
print("Actual default:", y_test.iloc[high_risk_position])

Selected test-set position: 96
Default probability: 0.9367
Risk score: 94
Risk category: High
Actual default: 1


In [37]:
local_shap = pd.DataFrame({
    'feature': feature_names,
    'feature_value': X_test_shap[high_risk_position],
    'shap_value': shap_values.values[high_risk_position]
})

local_shap['absolute_shap'] = np.abs(local_shap['shap_value'])

local_shap = (
    local_shap
    .sort_values('absolute_shap', ascending=False)
    .reset_index(drop=True)
)

print("TOP 15 CONTRIBUTORS FOR SELECTED HIGH-RISK APPLICATION")
print("=" * 90)

print(
    local_shap[
        ['feature', 'feature_value', 'shap_value']
    ].head(15).to_string(index=False)
)

TOP 15 CONTRIBUTORS FOR SELECTED HIGH-RISK APPLICATION
                      feature  feature_value  shap_value
    num__debt_to_income_ratio       2.872195    1.394535
            num__credit_score      -3.495010    0.990062
          num__loan_to_income       8.238634   -0.716738
      num__num_defaults_prior       4.578032    0.377549
        num__requested_amount       1.431193    0.242567
num__applicant_monthly_income      -1.292911    0.162933
           num__annual_income      -1.292933    0.162714
           num__interest_rate      -1.018563    0.108367
cat__employment_type_Salaried       0.000000    0.096964
        num__loan_term_months       0.841697   -0.091061
  num__total_outstanding_debt      -0.830218   -0.083040
        num__total_outflow_6m      -1.120987    0.073578
      cat__education_Graduate       1.000000    0.072882
           num__debt_to_limit       1.505233    0.063968
              num__dependents      -1.175401    0.051229


### Local SHAP Explanation — High-Risk Application

- A High-Risk application with a predicted default probability of **93.67%** and a risk score of **94** was selected for individual explanation.
- The applicant's actual outcome was also **default (1)**.
- **Debt-to-income ratio** was the strongest positive contributor to the predicted default risk for this application.
- **Credit score** was another major positive contributor to the model's default prediction.
- **Previous defaults**, requested loan amount, income-related variables, and interest rate also pushed the prediction toward higher default risk.
- **Loan-to-income ratio** produced the strongest negative SHAP contribution for this application, partially reducing the predicted risk.
- Loan term and total outstanding debt also contributed negatively to the prediction.
- Positive SHAP values push the prediction toward higher estimated default risk, while negative SHAP values push it toward lower estimated risk.
- Numerical feature values shown in the SHAP table are standardized model inputs rather than the original raw values.
- Overall, SHAP provides a transparent explanation of why this particular application received a High-Risk classification.

## 14. Save Final Model and Outputs

In [38]:
import joblib

joblib.dump(
    final_model,
    '/content/credora_final_model.pkl'
)

print("Final model saved successfully:")
print("/content/credora_final_model.pkl")

Final model saved successfully:
/content/credora_final_model.pkl


In [39]:
# Save risk-scoring results
risk_results.to_csv(
    '/content/credora_risk_results.csv',
    index=False
)

# Save global SHAP importance
shap_importance.to_csv(
    '/content/credora_shap_importance.csv',
    index=False
)

print("Files saved successfully:")
print("/content/credora_risk_results.csv")
print("/content/credora_shap_importance.csv")

Files saved successfully:
/content/credora_risk_results.csv
/content/credora_shap_importance.csv


In [40]:
from google.colab import files

files.download('/content/credora_final_model.pkl')
files.download('/content/credora_risk_results.csv')
files.download('/content/credora_shap_importance.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 15. Model Building Summary

### Modeling Approach
- The final modeling dataset contained **3,403 loan applications** with known default outcomes and **41 predictor features**.
- The data was split into **80% training and 20% testing sets** using stratification to preserve the default-class distribution.
- Numerical features were processed using median imputation and standardization.
- Categorical features were processed using constant-value imputation and one-hot encoding.
- Four baseline models were evaluated: **Logistic Regression, Decision Tree, Random Forest, and XGBoost**.
- **5-fold stratified cross-validation** was used on the training dataset for model comparison.

### Class Imbalance and Model Tuning
- The training data contained **2,084 non-default cases and 638 default cases**, producing an imbalance ratio of approximately **3.27:1**.
- Class weighting was evaluated to improve detection of the minority default class.
- Balanced Logistic Regression provided the strongest improvement in recall and F1-score.
- Hyperparameter tuning selected **C = 0.01** for Logistic Regression.
- The tuned model achieved a cross-validation **ROC-AUC of 0.7402** and **Recall of 65.52%**.
- Threshold optimization using out-of-fold training probabilities selected **0.45** as the operating threshold.
- At this threshold, cross-validated recall reached **75.08%** with an F1-score of **0.5042**.

### Final Held-Out Test Performance
- **Accuracy:** 63.14%
- **Precision:** 35.35%
- **Recall:** 69.81%
- **F1-Score:** 0.4693
- **ROC-AUC:** 0.7119
- The final model correctly identified **111 of 159 actual default cases**.
- Final recall was very close to the project target of 70%, while ROC-AUC remained below the target of 0.80.

### Risk Scoring
- Predicted default probabilities were converted into **0–100 risk scores**.
- Applications were classified as:
  - **Low Risk:** 0–33
  - **Medium Risk:** 34–66
  - **High Risk:** 67–100
- Among the held-out applications, **218 were Low Risk, 370 Medium Risk, and 93 High Risk**.
- Observed default rates increased from **10.55% in Low Risk**, to **24.59% in Medium Risk**, and **48.39% in High Risk**, supporting the usefulness of the risk segmentation.

### Explainability
- SHAP was used for both global and individual prediction explanations.
- The most influential global features included **debt-to-income ratio, credit score, 30-day late payments, credit inquiries, requested loan amount, income, interest rate, and loan term**.
- A selected High-Risk application received a **93.67% predicted probability of default and a risk score of 94**, and its actual outcome was also default.
- Local SHAP analysis identified the features that pushed this individual prediction toward higher or lower estimated default risk.

### Saved Outputs
- `credora_final_model.pkl`
- `credora_risk_results.csv`
- `credora_shap_importance.csv`

The modeling pipeline therefore provides probability-of-default prediction, a business-friendly risk score, risk segmentation, and explainable model outputs for credit-risk decision support.